# Central Forces: Kepler Problem and Orbital Mechanics

**MechanicsDSL Notebook 04** | Classical Mechanics | Advanced

The Kepler problem is the archetype of central force motion. This notebook derives orbits from the Lagrangian, identifies two conserved quantities via Noether's theorem, and demonstrates elliptic, parabolic, and hyperbolic trajectories.

**MechanicsDSL automatically identifies:**
- $\partial L/\partial t = 0$ → Energy conserved
- $\partial L/\partial \phi = 0$ (cyclic) → Angular momentum $L_z = mr^2\dot{\phi}$ conserved

**DSL specification:**
```
\system{kepler}
\parameter{M}{1.0}{kg}
\parameter{m}{1.0}{kg}
\parameter{G}{1.0}{1}
\lagrangian{0.5*m*(\dot{r}^2 + r^2*\dot{phi}^2) + G*M*m/r}
\target{python_numpy}
```

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 11
GM = 1.0  # gravitational parameter (G*M)

In [ ]:
def kepler_eom(t, y):
    """Kepler problem in Cartesian coordinates.
    Euler-Lagrange applied to L = 0.5*m*(xdot^2+ydot^2) + GM*m/r
    State: [x, y, vx, vy]
    """
    x, y, vx, vy = y
    r3 = (x**2+y**2)**1.5
    return [vx, vy, -GM*x/r3, -GM*y/r3]

def energy(y): return 0.5*(y[2]**2+y[3]**2) - GM/np.sqrt(y[0]**2+y[1]**2)
def ang_mom(y): return y[0]*y[3] - y[1]*y[2]  # L_z = x*vy - y*vx

# Orbital cases: e<1 ellipse, e=1 parabola, e>1 hyperbola
cases = [
    ('Ellipse (e=0.5)',    [1.0, 0.0, 0.0,  0.866], 'tab:blue'),
    ('Circle (e=0)',       [1.0, 0.0, 0.0,  1.0],   'tab:green'),
    ('Hyperbola (e=1.5)',  [1.0, 0.0, 0.0,  1.5],   'tab:red'),
]

fig, axes = plt.subplots(1, 2, figsize=(13,6))
fig.suptitle('Kepler Problem — Orbital Trajectories', fontweight='bold')

for label, ic, color in cases:
    sol = solve_ivp(kepler_eom, [0, 20], ic, max_step=0.01, rtol=1e-10, atol=1e-12,
                    events=[lambda t,y: np.sqrt(y[0]**2+y[1]**2)-5.0])
    axes[0].plot(sol.y[0], sol.y[1], color=color, lw=1.2, label=label)
    E = np.array([energy(sol.y[:,i]) for i in range(sol.y.shape[1])])
    L = np.array([ang_mom(sol.y[:,i]) for i in range(sol.y.shape[1])])
    axes[1].semilogy(sol.t, np.abs((E-E[0])/(np.abs(E[0])+1e-15)), color=color, lw=0.7, ls='-', label=f'{label} energy')

axes[0].plot(0, 0, 'yo', ms=8, label='Focus (M)')
axes[0].set_aspect('equal'); axes[0].set_xlabel('x'); axes[0].set_ylabel('y')
axes[0].legend(fontsize=9); axes[0].set_title('Trajectories')
axes[1].set_xlabel('Time'); axes[1].set_ylabel(r'$|\Delta E/E_0|$')
axes[1].set_title('Energy conservation (Noether)'); axes[1].legend(fontsize=8)
plt.tight_layout(); plt.show()

In [ ]:
# Verify both conserved quantities for circular orbit
ic = [1.0, 0.0, 0.0, 1.0]
t_eval = np.linspace(0, 10*2*np.pi, 5000)
sol = solve_ivp(kepler_eom, [0, t_eval[-1]], ic, t_eval=t_eval, method='DOP853', rtol=1e-12, atol=1e-14)
E_arr = np.array([energy(sol.y[:,i]) for i in range(sol.y.shape[1])])
L_arr = np.array([ang_mom(sol.y[:,i]) for i in range(sol.y.shape[1])])
print('Noether conservation check (circular orbit, 10 periods):')
print(f'  Energy drift:          |dE/E0| = {np.max(np.abs((E_arr-E_arr[0])/E_arr[0])):.2e}')
print(f'  Angular momentum drift:|dL/L0| = {np.max(np.abs((L_arr-L_arr[0])/L_arr[0])):.2e}')

## Summary
| Orbit type | Eccentricity | Energy |
|------------|-------------|--------|
| Circle | $e=0$ | $E = -GM/2r$ |
| Ellipse | $0 < e < 1$ | $E < 0$ |
| Parabola | $e=1$ | $E = 0$ |
| Hyperbola | $e > 1$ | $E > 0$ |

Both $E$ and $L_z$ are conserved (Noether: time-translation and rotational symmetry). MechanicsDSL identifies both automatically from the Lagrangian.

**Next:** [05 — Hamiltonian Mechanics and Phase Space](05_hamiltonian.ipynb)